# Misc Features Pipeline

Loads raw HDB resale and RPI data, merges them, computes real prices, filters to 2021-Q1 onwards, and saves the base merged dataset.

## Inputs
- `../../data/HDBResalePriceIndex1Q2009100Quarterly.csv`
- `../../data/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv`

## Outputs
- `../../merged_data/merged_hdb_resale_with_rpi.csv` — merged dataset with RPI and real prices, filtered to 2021-Q1+

In [15]:
import pandas as pd
import numpy as np

# Load the datasets
print("Loading datasets...")
rpi_df = pd.read_csv('../../data/HDBResalePriceIndex1Q2009100Quarterly.csv')
resale_df = pd.read_csv('../../data/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv')

print(f"RPI dataset shape: {rpi_df.shape}")
print(f"Resale prices dataset shape: {resale_df.shape}")

Loading datasets...
RPI dataset shape: (144, 2)
Resale prices dataset shape: (228126, 11)


In [16]:
# Step 1: Convert the 'month' column in resale_df to datetime
print("\nConverting month to datetime format...")
resale_df['month'] = pd.to_datetime(resale_df['month'])


Converting month to datetime format...


In [17]:
# Step 2: Create a 'quarter' column in resale_df in format "YYYY-QX"
print("Creating quarter column from monthly data...")
resale_df['year'] = resale_df['month'].dt.year
resale_df['q'] = resale_df['month'].dt.quarter
resale_df['quarter'] = resale_df['year'].astype(str) + '-Q' + resale_df['q'].astype(str)
resale_df = resale_df.drop(columns=['year', 'q'])

Creating quarter column from monthly data...


In [18]:
# Step 3: Merge the datasets
print("\nMerging datasets on 'quarter' column...")
merged_df = resale_df.merge(rpi_df, on='quarter', how='left')
merged_df = merged_df.rename(columns={'index': 'rpi'})

print(f"Merged dataset shape: {merged_df.shape}")


Merging datasets on 'quarter' column...
Merged dataset shape: (228126, 13)


In [19]:
# Step 4: Adjust resale prices by RPI
# Set base period to 2025 Q4
BASE_RPI = 203.6  # 2025 Q4 RPI value — real prices expressed in Q4 2025 SGD

# Impute missing RPI with BASE_RPI (applies to unmatched quarters, e.g. 2026-Q1)
# Real price = nominal price when rpi == BASE_RPI, which is a valid approximation
# for quarters whose RPI hasn't been published yet but is expected to be near the base.
missing_rpi = merged_df['rpi'].isna().sum()
if missing_rpi > 0:
    print(f"Rows with missing RPI (imputed to BASE_RPI={BASE_RPI}): {missing_rpi:,}")
    print(f"  Quarters imputed: {sorted(merged_df[merged_df['rpi'].isna()]['quarter'].unique())}")
    merged_df['rpi'] = merged_df['rpi'].fillna(BASE_RPI)

merged_df['resale_price_real'] = merged_df['resale_price'] * (BASE_RPI / merged_df['rpi'])
merged_df['resale_price_real'] = merged_df['resale_price_real'].round(2)

print(f"Total rows retained: {len(merged_df):,}")

Rows with missing RPI (imputed to BASE_RPI=203.6): 6,058
  Quarters imputed: ['2026-Q1']
Total rows retained: 228,126


In [20]:
# Step 4b: Derived features
import re

# remaining_lease_years: parse "61 years 04 months" → 61.33
def parse_remaining_lease(val):
    if pd.isna(val) or val == '':
        return np.nan
    m = re.match(r'(\d+)\s*year', str(val))
    if not m:
        return np.nan
    years = int(m.group(1))
    m2 = re.search(r'(\d+)\s*month', str(val))
    months = int(m2.group(1)) if m2 else 0
    return round(years + months / 12, 4)

merged_df['remaining_lease_years'] = merged_df['remaining_lease'].apply(parse_remaining_lease)

# floor_category: extract lower bound of storey range, bucket into Low/Mid/High
def floor_category(storey_range):
    if pd.isna(storey_range):
        return np.nan
    m = re.match(r'(\d+)', str(storey_range))
    if not m:
        return np.nan
    lower = int(m.group(1))
    if lower <= 5:
        return 'Low'
    elif lower <= 12:
        return 'Mid'
    else:
        return 'High'

merged_df['floor_category'] = merged_df['storey_range'].apply(floor_category)

# year: integer year from month (for stratification)
merged_df['year'] = merged_df['month'].dt.year

print(f"remaining_lease_years — null: {merged_df['remaining_lease_years'].isna().sum():,}")
print(f"floor_category distribution:\n{merged_df['floor_category'].value_counts().to_string()}")
print(f"year range: {merged_df['year'].min()} – {merged_df['year'].max()}")

remaining_lease_years — null: 0
floor_category distribution:
floor_category
Low     92462
Mid     90426
High    45238
year range: 2017 – 2026


In [24]:
# Step 5: Filter to 2021-Q1 onwards (avoids COVID structural break) and save
merged_df = merged_df[merged_df['quarter'] >= '2021-Q1'].copy()
print(f"After 2021-Q1 filter: {len(merged_df):,} rows")

output_file = '../../merged_data/merged_hdb_resale_with_rpi.csv'
print(f"\nSaving merged dataset to {output_file}...")
merged_df.to_csv(output_file, index=False)
print("Done!")
print(f"\n=== SUMMARY ===")
print(f"Total resale transactions: {len(merged_df):,}")
print(f"Date range: {merged_df['month'].min()} to {merged_df['month'].max()}")
print(f"\nColumns in merged dataset:")
print(merged_df.columns.tolist())

After 2021-Q1 filter: 140,537 rows

Saving merged dataset to ../../merged_data/merged_hdb_resale_with_rpi.csv...
Done!

=== SUMMARY ===
Total resale transactions: 140,537
Date range: 2021-01-01 00:00:00 to 2026-03-01 00:00:00

Columns in merged dataset:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price', 'quarter', 'rpi', 'resale_price_real', 'remaining_lease_years', 'floor_category', 'year']
